In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
import os
import gc
import shutil
warnings.filterwarnings('ignore')

SEED = 1 
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
from huggingface_hub import login

login()

In [ ]:
original_ds = load_dataset("ailsntua/QEvasion")
from sklearn.model_selection import train_test_split

original_train_full = original_ds['train'].to_pandas()
test_df = original_ds['test'].to_pandas()

train_df, val_df = train_test_split(
    original_train_full,
    test_size=700,
    random_state=SEED,
    stratify=original_train_full['clarity_label']
)

In [ ]:
HYPOTHESES = {
    "Clear Reply": "The speaker gave a clear reply.",
    "Ambivalent": "The speaker was ambivalent.",
    "Clear Non-Reply": "The speaker clearly did not reply."
}

clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [ ]:
def convert_to_nli_format(df):
    nli_data = []
    
    for _, row in df.iterrows():
        text = f"Question: {row['question']}\nAnswer: {row['interview_answer']}"
        true_label = row['clarity_label']
        
        for class_label, hypothesis in HYPOTHESES.items():
            label = 0 if class_label == true_label else 1
            
            nli_data.append({
                'text': text,
                'hypothesis': hypothesis,
                'label': label,
                'original_label': true_label  
            })
    
    return pd.DataFrame(nli_data)

train_nli_df = convert_to_nli_format(train_df)
val_nli_df = convert_to_nli_format(val_df)
test_nli_df = convert_to_nli_format(test_df)

In [ ]:
MODEL_NAME = "mlburnham/Political_DEBATE_large_v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "ENTAILMENT", 1: "NOT_ENTAILMENT"},
    label2id={"ENTAILMENT": 0, "NOT_ENTAILMENT": 1}
)

In [ ]:
def tokenize_nli(examples):
    return tokenizer(
        examples['text'],
        examples['hypothesis'],
        truncation=True,
        padding=False,
        max_length=512  
    )

train_dataset = Dataset.from_pandas(train_nli_df[['text', 'hypothesis', 'label']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_nli_df[['text', 'hypothesis', 'label']], preserve_index=False)

train_tokenized = train_dataset.map(
    tokenize_nli,
    batched=True,
    remove_columns=['text', 'hypothesis']
)

val_tokenized = val_dataset.map(
    tokenize_nli,
    batched=True,
    remove_columns=['text', 'hypothesis']
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStopping(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_nli_debate",
    
    learning_rate=9e-6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    warmup_ratio=0.06,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    
    bf16=True,  
    bf16_full_eval=True,
    optim="adamw_torch",
    max_grad_norm=1.0,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1, 
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_steps=50,
    logging_strategy="steps",
    report_to="none",
    
    dataloader_num_workers=4,  
    seed=SEED,
    group_by_length=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStopping(patience=3, metric_name="eval_f1", greater_is_better=True)],
)

trainer.train()

In [ ]:
def predict_nli(trainer, tokenizer, df):
    predictions = []
    
    for _, row in df.iterrows():
        text = f"Question: {row['question']}\nAnswer: {row['interview_answer']}"
        
        hypothesis_scores = {}
        
        for class_label, hypothesis in HYPOTHESES.items():
            inputs = tokenizer(
                text,
                hypothesis,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            with torch.no_grad():
                outputs = trainer.model(**inputs)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=-1)
                
                hypothesis_scores[class_label] = probs[0, 0].item()
        
        predicted_label = max(hypothesis_scores, key=hypothesis_scores.get)
        predictions.append(predicted_label)
    
    return predictions

trainer.model.to(device)
trainer.model.eval()

test_predictions = predict_nli(trainer, tokenizer, test_df)

In [ ]:
y_true = test_df['clarity_label'].tolist()
y_pred = test_predictions

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', labels=clarity_labels, zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=clarity_labels, zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=clarity_labels, zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print(f"\nPer-class F1 scores:")
for label, f1_val in zip(clarity_labels, per_class_f1):
    print(f"  {label}: {f1_val:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm)

In [ ]:
results_dict = {
    'accuracy': accuracy,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
    'per_class_f1': {label: f1_val for label, f1_val in zip(clarity_labels, per_class_f1)}
}

results_df = pd.DataFrame([{
    'Method': 'NLI (Political_DEBATE)',
    'Accuracy': accuracy,
    'Macro F1': macro_f1,
    'Weighted F1': weighted_f1,
    'Clear Reply F1': per_class_f1[0],
    'Ambivalent F1': per_class_f1[1],
    'Clear Non-Reply F1': per_class_f1[2]
}])

print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/nli_approach_results.csv', index=False)